# Policy-Aware Flood Risk Pipeline

End-to-end test: ODRL policy evaluation + encrypted data + urban_pfr pipeline.

1. Encrypt `buildings.fgb` with a new AES key
2. Fetch the ODRL policy from the nanopub
3. Evaluate the policy (purpose = AcademicResearch)
4. If approved, decrypt the file
5. Run the urban_pfr pipeline on the decrypted data

## Step 1: Encrypt buildings.fgb with a new key

In [ ]:
import os
from pathlib import Path
from fair_data_access.encrypt import encrypt_file, decrypt_file

BUILDINGS_PLAINTEXT = Path("Hamburg-data/buildings.fgb")
BUILDINGS_ENCRYPTED = Path("Hamburg-data/buildings.fgb.enc")

if BUILDINGS_ENCRYPTED.exists():
    # Reuse existing encrypted file and key
    print(f"Encrypted file already exists: {BUILDINGS_ENCRYPTED} ({BUILDINGS_ENCRYPTED.stat().st_size / 1e6:.1f} MB)")
    print("Enter the hex key from the previous run:")
    key_hex = input("Key (hex): ").strip()
    dataset_key = bytes.fromhex(key_hex)
    print(f"Key loaded ({len(dataset_key)} bytes)")
else:
    # Generate a new AES-256 key and encrypt
    dataset_key = os.urandom(32)
    print(f"Generated AES-256 key: {dataset_key.hex()}")
    print("SAVE THIS KEY — you will need it to decrypt and to update the GitHub Secret")
    encrypt_file(BUILDINGS_PLAINTEXT, BUILDINGS_ENCRYPTED, key=dataset_key)
    print(f"Encrypted: {BUILDINGS_ENCRYPTED} ({BUILDINGS_ENCRYPTED.stat().st_size / 1e6:.1f} MB)")

## Step 2: Fetch the ODRL policy from the nanopub

In [ ]:
import json
from fair_data_access.policy import fetch_policy

POLICY_NANOPUB = "https://w3id.org/np/RAir7keZs8Jy7i8HvRI7X4SmefJtY3jG2WgyFlsZa9-iw"

print(f"Fetching policy from: {POLICY_NANOPUB}")
policy = fetch_policy(POLICY_NANOPUB)
print(f"\nPolicy retrieved:")
print(json.dumps(policy, indent=2))

## Step 3: Evaluate the policy

In [ ]:
from fair_data_access.policy import evaluate_policy

# Simulate a researcher requesting access for academic research
REQUESTER_DID = "did:web:fair2adapt.github.io:fair-data-access"
PURPOSE = "AcademicResearch"

print(f"Requester: {REQUESTER_DID}")
print(f"Purpose:   {PURPOSE}")
print()

access_granted = evaluate_policy(policy, requester_did=REQUESTER_DID, purpose=PURPOSE)

if access_granted:
    print("ACCESS GRANTED - policy permits this use")
else:
    print("ACCESS DENIED - policy does not permit this use")
    raise SystemExit("Cannot proceed without access")

In [ ]:
# Test: commercial purpose should be denied
denied = evaluate_policy(policy, requester_did=REQUESTER_DID, purpose="CommercialUse")
print(f"Commercial purpose: {'GRANTED' if denied else 'DENIED'} (expected: DENIED)")

## Step 4: Decrypt the file

In [ ]:
import tempfile
from pathlib import Path

# Decrypt to a temporary file
decrypted_path = Path(tempfile.mkdtemp()) / "buildings.fgb"
decrypt_file(BUILDINGS_ENCRYPTED, decrypted_path, key=dataset_key)

print(f"Decrypted: {decrypted_path} ({decrypted_path.stat().st_size / 1e6:.1f} MB)")

# Verify it matches the original
original_size = BUILDINGS_PLAINTEXT.stat().st_size
decrypted_size = decrypted_path.stat().st_size
print(f"Original:  {original_size / 1e6:.1f} MB")
print(f"Match: {original_size == decrypted_size}")

## Step 5: Run the urban_pfr pipeline

In [ ]:
import yaml
from urban_pfr.analyzer import PFRAnalyzer

# Load config and override buildings path with decrypted file
with open("config_hamburg_full.yaml") as f:
    config = yaml.safe_load(f)

# Point to the decrypted buildings file
config["project"]["paths"]["buildings"] = str(decrypted_path)

print(f"Pipeline config:")
print(f"  Buildings: {config['project']['paths']['buildings']} (decrypted)")
print(f"  Streets:   {config['project']['paths']['streets']}")
print(f"  Floods:    {config['project']['paths']['flood_dir']}")
print(f"  Output:    {config['project']['paths']['output_dir']}")
print()

# Run the pipeline
analyzer = PFRAnalyzer(config)
analyzer.load_data()
analyzer.run_pipeline()

# Save results
analyzer.save_results()

print("\nPipeline completed successfully!")

## Cleanup

In [ ]:
import shutil

# Remove decrypted temp file
shutil.rmtree(decrypted_path.parent)
print(f"Removed temp decrypted file")

# Remove local encrypted copy (the real one is on S3)
BUILDINGS_ENCRYPTED.unlink()
print(f"Removed local encrypted copy")

print(f"\nKey for updating GitHub Secret and S3:")
print(f"  {dataset_key.hex()}")